In [ ]:
import sys
import os

import torch
from torch.optim import SGD, AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, CosineAnnealingWarmRestarts
from torch.nn import CrossEntropyLoss
from torchsummary import summary

project_root = os.path.abspath(os.path.join(os.path.dirname(__file__), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

import src.dataloader as dataloader
from src.dataloader import PeopleDataset

from src.utils.engine import setup_trainer, setup_evaluators, train_epoch_and_get_metrics_dict, calculate_epoch_metrics
from src.utils.logging import setup_metrics_history, add_metrics_to_history, print_epoch_summary, save_best_models
from src.utils import plotting

In [ ]:
# train function (from file src/launcher/traner.py)

def train_model(config, model_class, class_exclusion_threshold=None, classes_to_exclude=None):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}\n")

    """Preparing the data"""
    train_transforms = dataloader.get_transforms(augmentation_type=config.TRAIN_AUGMENTATION_TYPE)
    valid_transforms = dataloader.get_transforms(augmentation_type=config.VALID_AUGMENTATION_TYPE)

    print("Loading the dataset...")
    full_dataset = dataloader.PeopleDataset(config.PATH_TO_DATA)
    full_dataset.print_class_distribution()

    if class_exclusion_threshold:
        print("Removing rare classes")
        # Option 1: Filter by minimum threshold of class in dataset
        full_dataset.filter_by_min_threshold(min_threshold=class_exclusion_threshold)

    if classes_to_exclude:
        print("Excluding selected classes")
        # Option 2: Filter by explicitly excluding class names
        full_dataset.filter_by_classes(classes_to_exclude=classes_to_exclude)

    if classes_to_exclude or class_exclusion_threshold:
        # Rebuild class_to_index AFTER filtering
        full_dataset.class_names = sorted(
            list(set(full_dataset.labels)))  # Get unique remaining labels (which are strings) and sort them
        full_dataset.class_to_index = {cls_name: i for i, cls_name in enumerate(full_dataset.class_names)}
        print(f"Number of classes after filtering: {len(full_dataset.class_names)}")  # Verify the number of classes

    train_set, valid_set = dataloader.split_dataset(full_dataset, valid_ratio=0.2)
    train_set.dataset.transform = train_transforms
    valid_set.dataset.transform = valid_transforms

    # Showing first 12 images after transforming them
    # plotting.show_first_images(full_dataset)

    print(f"Setting up data loaders with batch_size={config.BATCH_SIZE}...")
    train_loader, valid_loader = dataloader.setup_data_loaders(
        batch_size=config.BATCH_SIZE,
        train_set=train_set,
        valid_set=valid_set
    )

    """Training setup"""
    num_classes = len(full_dataset.class_names)  # Use the updated class_names
    print(f"Number of classes: {num_classes}")

    model = model_class(num_classes=num_classes)
    model.to(device)
    summary(model, (3, 288, 512))
    print("\n")

    train_indices = train_set.indices
    train_targets = [full_dataset[idx][1] for idx in train_indices]
    class_counts = [train_targets.count(i) for i in range(num_classes)]
    class_counts = [c if c != 0 else 1 for c in class_counts]
    class_weights = 1.0 / torch.tensor(class_counts, dtype=torch.float)
    class_weights = class_weights / class_weights.sum()
    class_weights = class_weights.to(device)

    optimizer = AdamW(model.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)

    if config.SCHEDULER == 'CosineAnnealingWarmRestarts':
        scheduler = CosineAnnealingWarmRestarts(optimizer=optimizer, T_0=25, eta_min=1e-6)
    elif config.SCHEDULER == 'CosineAnnealingLR':
        scheduler = CosineAnnealingLR(optimizer=optimizer, T_max=config.NUM_EPOCHS, eta_min=0)

    criterion = CrossEntropyLoss(weight=class_weights.to(device))

    print("Initializing trainer and evaluators...")
    trainer = setup_trainer(model, optimizer, criterion, device)
    train_evaluator, valid_evaluator = setup_evaluators(model, criterion, device)

    train_metrics_history, valid_metrics_history = setup_metrics_history()

    best_valid_loss = float('inf')
    best_valid_f1 = 0.0
    model_name = model.__class__.__name__

    print(f"\nStarting training for {config.NUM_EPOCHS} epochs...")

    for epoch in range(config.NUM_EPOCHS):
        print(f"\nEpoch {epoch + 1}/{config.NUM_EPOCHS}")

        train_metrics_dict = train_epoch_and_get_metrics_dict(model, train_loader, criterion, optimizer, device, epoch,
                                                              config.NUM_EPOCHS)
        scheduler.step()
        add_metrics_to_history(train_metrics_history, train_metrics_dict)

        valid_metrics_dict = {}
        if valid_loader:
            valid_metrics_dict = calculate_epoch_metrics(model, valid_loader, criterion, device)
            add_metrics_to_history(valid_metrics_history, valid_metrics_dict)
            best_valid_loss, best_valid_f1 = save_best_models(
                current_metrics=valid_metrics_dict,
                model=model,
                model_name=model_name,
                best_loss=best_valid_loss,
                best_f1=best_valid_f1,
                save_dir=config.SAVE_DIR
            )

        print_epoch_summary(epoch, train_metrics_dict, valid_metrics_dict)

    """Results visualization"""
    print("\nTraining completed!")
    print(f"Results location: {config.RESULT_DIR}")
    print("Saving results...")
    metrics_to_plot = ['accuracy', 'precision', 'recall', 'f1', 'loss']
    plotting.plot_metrics(train_metrics_history, valid_metrics_history, metrics_to_plot, save_path=config.RESULT_DIR)

    # To plot loss and one metric
    # plot_metric_and_loss(train_metrics_history, valid_metrics_history, "accuracy")

    class_names = full_dataset.class_names  # Use the updated class_names
    # plotting.visualize_predictions(model, valid_loader, device, class_names)

    print(f"\nPlotting metrics per class...")
    plotting.plot_metrics_per_class(model, valid_loader, device, class_names, save_path=config.RESULT_DIR)

    # evaluate_model(model, test_loader, criterion, device)


In [ ]:
import torch.nn as nn


class PoseCNNsc_13_24_35(nn.Module):
    def __init__(self, num_classes=20):
        super(PoseCNNsc_13_24_35, self).__init__()

        def conv_block(in_channels, out_channels, use_dropout=False):
            layers = [
                nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2)
            ]
            if use_dropout:
                layers.append(nn.Dropout2d(0.2))
            return nn.Sequential(*layers)

        self.conv1 = conv_block(3, 32)  # Output: (32, H/2, W/2)
        self.conv2 = conv_block(32, 64)  # Output: (64, H/4, W/4)
        self.conv3 = conv_block(64, 128, use_dropout=False)  # Output: (128, H/8, W/8)
        self.conv4 = conv_block(128, 256, use_dropout=True)  # Output: (256, H/16, W/16)
        self.conv5 = conv_block(256, 512, use_dropout=True)  # Output: (512, H/32, W/32)

        # Skip connection from conv1 to conv3
        self.skip1_proj = nn.Sequential(
            nn.Conv2d(32, 128, kernel_size=1),
            nn.MaxPool2d(kernel_size=4, stride=4)  # Downsample H/2 -> H/8, W/2 -> W/8
        )

        # Skip connection from conv2 to conv4
        self.skip2_proj = nn.Sequential(
            nn.Conv2d(64, 256, kernel_size=1),
            nn.MaxPool2d(kernel_size=2, stride=2)  # Downsample H/4 -> H/8, W/4 -> W/8
        )

        # Skip connection from conv3 to conv5
        self.skip3_proj = nn.Sequential(
            nn.Conv2d(128, 512, kernel_size=1),
            nn.MaxPool2d(kernel_size=4, stride=4)  # Downsample H/8 -> H/32, W/8 -> W/32
        )

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x1 = self.conv1(x)  # (32, H/2, W/2)
        x2 = self.conv2(x1)  # (64, H/4, W/4)
        x3 = self.conv3(x2)  # (128, H/8, W/8)

        # Skip connection 1 -> 3
        skip1 = self.skip1_proj(x1)  # (128, H/8, W/8)
        x3 = x3 + skip1

        # Skip connection 2 -> 4
        skip2 = self.skip2_proj(x2)  # (256, H/8, W/8)
        x4 = self.conv4(x3)  # (256, H/16, W/16)
        # Необходимо выполнить еще одно понижение разрешения для skip2, чтобы соответствовать x4
        skip2_downsampled = nn.MaxPool2d(kernel_size=2, stride=2)(skip2)  # (256, H/16, W/16)
        x4 = x4 + skip2_downsampled

        x5 = self.conv5(x4)  # (512, H/32, W/32)

        # Skip connection 3 -> 5
        skip3 = self.skip3_proj(x3)  # (512, H/32, W/32)
        x5 = x5 + skip3

        x_pooled = self.pool(x5)  # (512, 1, 1)
        return self.classifier(x_pooled)


In [ ]:
# simulating config file as class

class Config:
    PATH_TO_DATA = "PATH_TO_YOUR_DATA"
    RESULT_DIR = "RESULT_DIR"
    SAVE_DIR = "WEIGHTS_SAVE_DIR"

    BATCH_SIZE = 32
    LEARNING_RATE = 0.001
    MOMENTUM = 0.9
    NUM_EPOCHS = 75

    SCHEDULER = 'CosineAnnealingLR'

    WEIGHT_DECAY = 3e-3

    TRAIN_AUGMENTATION_TYPE = "basic"
    VALID_AUGMENTATION_TYPE = None

config = Config()

SyntaxError: invalid syntax (<ipython-input-1-4dde57f871c2>, line 1)

In [ ]:
model_to_train = PoseCNNsc_13_24_35

# Call the training function from trainer.py
trainer.train_model(config, model_to_train)